# TVM Relay 自动量化校准

参考：`tvm/src/relay/quantize/calibrate.cc` 和 `tvm/python/tvm/relay/quantize/_calibrate.py`

In [1]:
import set_env

In [2]:
import tvm
from tvm.relay import analysis as _analysis
from tvm.relay import op as _op
from tvm.relay.op import op as _reg
from tvm.relay import expr as _expr

In [3]:
@_op.register_compute("simulated_quantize")
def simulated_quantize_compute(attrs, inputs, out_type):
    """模拟量化计算实现

    该函数实现了模拟量化操作，包括对称量化和非对称量化，
    并模拟了量化过程中的舍入误差。

    Parameters
    ----------
    attrs : Attrs
        量化属性，包含kind、sign、rounding等参数
    inputs : list of Tensor
        输入张量列表，包含data、scale、clip_min、clip_max和可选的zero_point
    out_type : Type
        输出类型

    Returns
    -------
    list of Tensor
        量化后并恢复的张量
    """
    assert len(inputs) == 5, "输入张量数量必须为5"
    assert attrs.sign, "仅支持有符号量化"
    assert attrs.rounding == "round", "仅支持四舍五入舍入模式"

    data = inputs[0]         # 输入数据
    scale = inputs[1]        # 缩放因子
    clip_min = inputs[2]     # 裁剪最小值
    clip_max = inputs[3]     # 裁剪最大值
    zero_point = inputs[4] if len(inputs) > 4 else None  # 零点(非对称量化使用)

    # 恒等映射模式，直接返回原始数据
    if attrs.kind == QAnnotateKind.IDENTITY:
        return [topi.identity(data)]

    # 模拟量化舍入误差
    if zero_point is not None and attrs.kind in [QAnnotateKind.ACTIVATION, QAnnotateKind.INPUT]:
        # 非对称量化: (x/s + zp) -> 量化 -> (q - zp) * s
        scaled_data = topi.divide(data, scale)          # 数据除以缩放因子
        shifted_data = topi.add(scaled_data, zero_point) # 添加零点偏移
        clipped_data = topi.maximum(topi.minimum(shifted_data, clip_max), clip_min)  # 裁剪到有效范围
        round_data = topi.round(clipped_data)           # 四舍五入量化
        zero_shifted_data = topi.subtract(round_data, zero_point)  # 减去零点偏移
        rdata = topi.multiply(zero_shifted_data, scale)  # 乘以缩放因子恢复
    else:
        # 对称量化: x/s -> 量化 -> q * s
        scaled_data = topi.divide(data, scale)          # 数据除以缩放因子
        clipped_data = topi.maximum(topi.minimum(scaled_data, clip_max), clip_min)  # 裁剪到有效范围
        round_data = topi.round(clipped_data)           # 四舍五入量化
        rdata = topi.multiply(round_data, scale)         # 乘以缩放因子恢复
    
    return [rdata]


_reg.register_injective_schedule("simulated_quantize")
_reg.register_pattern("simulated_quantize", _reg.OpPattern.ELEMWISE)

def simulated_quantize(
    data,
    dom_scale, 
    zero_point, 
    clip_min, 
    clip_max,):
    return _expr.Call(
        _op.get("simulated_quantize"), dom_scale, 
        zero_point, 
        clip_min, 
        clip_max, 
        # kind
    )

In [4]:
# %0 = relay.op.annotation.simulated_quantize(%x, %dom_scale, %zero_point, %clip_min, %clip_max, kind=1);
#   %1 = relay.op.annotation.simulated_quantize(meta[relay.Constant][0] /* ty=Tensor[(16, 3, 3, 3), float32] span=node_conv2d.conv.weight:0:0 */, %dom_scale1, %zero_point1, %clip_min1, %clip_max1, kind=2, mode="i8_channel");
#   %2 = nn.conv2d(%0, %1, padding=[1, 1, 1, 1], channels=16, kernel_size=[3, 3]);
#   %3 = relay.op.annotation.simulated_quantize(meta[relay.Constant][1] /* ty=Tensor[(16, 1, 1), float32] */, %dom_scale2, %zero_point2, %clip_min2, %clip_max2, kind=2);
#   %4 = add(%2, %3);
#   %5 = relay.op.annotation.simulated_quantize(%4, %dom_scale3, %zero_point3, %clip_min3, %clip_max3, kind=1);
#   %6 = annotation.cast_hint(%5, dtype="int8");
#   annotation.stop_fusion(%6)

In [5]:
from tvm import relay

In [6]:
x = relay.var("x", shape=(1, 3, 8, 8), dtype="float32")
# dom_scale: float32, %zero_point: int32, %clip_min: float32, %clip_max: float32,
dom_scale = relay.var("dom_scale", shape=(), dtype="float32")
zero_point = relay.var("zero_point", shape=(), dtype="int32")
clip_min = relay.var("clip_min", shape=(), dtype="float32")
clip_max = relay.var("clip_max", shape=(), dtype="float32")
x = simulated_quantize(x, dom_scale, zero_point, clip_min, clip_max)

TVMError: Traceback (most recent call last):
  3: _ZN3tvm7runtime13PackedFun
  2: tvm::runtime::TypedPackedFunc<tvm::relay::Call (tvm::RelayExpr, tvm::runtime::Array<tvm::RelayExpr, void>, tvm::Attrs, tvm::runtime::Array<tvm::Type, void>, tvm::Span)>::AssignTypedLambda<tvm::relay::__mk_TVM17::{lambda(tvm::RelayExpr, tvm::runtime::Array<tvm::RelayExpr, void>, tvm::Attrs, tvm::runtime::Array<tvm::Type, void>, tvm::Span)#1}>(tvm::relay::__mk_TVM17::{lambda(tvm::RelayExpr, tvm::runtime::Array<tvm::RelayExpr, void>, tvm::Attrs, tvm::runtime::Array<tvm::Type, void>, tvm::Span)#1}, std::__cxx11::basic_string<char, std::char_traits<char>, std::allocator<char> >)::{lambda(tvm::runtime::TVMArgs const&, tvm::runtime::TVMRetValue*)#1}::operator()(tvm::runtime::TVMArgs const&, tvm::runtime::TVMRetValue*) const
  1: tvm::runtime::TVMMovableArgValueWithContext_::operator tvm::Span<tvm::Span>() const
  0: _ZN3tvm7runtime6deta
  4: _ZN3tvm7runtime13PackedFun
  3: tvm::runtime::TypedPackedFunc<tvm::rela

In [ ]:
RewriteDataflowReshape